# Chapter 23 Companion Notebook: Sentiment and Text Classification

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch23_Sentiment_and_Text_Classification.ipynb)

This notebook accompanies Chapter 23 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




This notebook is designed as a classroom appendix. It uses synthetic customer feedback data so that every step can run safely in Google Colab without external files, paid APIs, or private customer information. The goal is not to make sentiment analysis look automatic. The goal is to show how analysts define a label, evaluate text classifiers, set business thresholds, review errors, and turn message-level predictions into decision-ready indicators.

## Why this matters (business framing)

Sentiment analysis is often introduced as a simple task: classify text as positive, neutral, or negative. In business analytics, that framing is too narrow. A sentiment model is a measurement instrument. Its outputs can support monitoring, complaint triage, service recovery, product improvement, and downstream prediction, but only if the task definition, labeling rules, validation strategy, and reporting unit match the business decision.

This notebook treats sentiment and text classification as a supervised measurement workflow. We will compare rating-based proxy labels with text-based labels, examine the unit of analysis, simulate annotator disagreement, build a lexicon baseline, train sparse and dense text classifiers, evaluate threshold choices, review operational errors, audit leakage, and aggregate predictions into dashboards and downstream churn features.

## Agenda

1. Setup and reproducibility
2. Synthetic customer feedback corpus with metadata
3. Sentiment as a measurement task
4. Ratings as imperfect proxy labels
5. Unit of analysis and target of opinion
6. Label agreement and data quality controls
7. Lexicon and rule-based sentiment baselines
8. Supervised text classification with sparse features
9. Dense representation baseline
10. Evaluation, thresholds, calibration, and triage policy
11. Slice evaluation, error review, and leakage audit
12. Dashboards, downstream analytics, governance outputs, and exercises

## Connection map

Chapter 20 covered text pre-processing and data quality. Chapter 21 introduced embeddings and similarity. Chapter 22 showed how recurring themes can be discovered from text. Chapter 23 uses those foundations to build supervised sentiment and text classification systems. The practical workflow is to define a measurable label, create reliable training data, choose a transparent baseline, evaluate with business risk in mind, and report predictions at the level where managers actually make decisions.

Chapter 24 will extend this discussion to transformer-based NLP and foundation models. The same design rules still apply there: a more powerful representation does not remove the need for clear labels, leakage-safe evaluation, error review, and governance.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# - install missing packages if needed
# - import libraries
# - set seeds
# - configure output folders
# ============================================================

import os
# Keep classroom notebooks lightweight and prevent BLAS/OpenMP oversubscription in small CPU environments.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}

for import_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import math
import random
import re
import warnings
from collections import Counter, defaultdict
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("/content/ch23_outputs") if Path("/content").exists() else Path("ch23_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

print("Environment ready")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## Utility functions

These helper functions keep the main sections focused on measurement and decision logic. They support text normalization, compact profiling, confusion-matrix displays, threshold analysis, slice-level evaluation, and business-facing error review.

In [ ]:
# ============================================================
# Utility functions for text classification and evaluation
# ============================================================

CUSTOM_STOP_WORDS = set(ENGLISH_STOP_WORDS).union({
    "customer", "customers", "product", "products", "item", "items", "experience",
    "really", "just", "like", "got", "get", "use", "used", "using", "overall",
})


def normalize_text(text):
    # Lowercase text, remove noisy symbols, and collapse whitespace.
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s'!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def simple_tokenize(text, min_len=2):
    tokens = re.findall(r"[a-z][a-z']+", normalize_text(text))
    return [tok for tok in tokens if len(tok) >= min_len and tok not in CUSTOM_STOP_WORDS]


def compact_profile(df, rows=8):
    # Display a compact profile of the corpus and a few example records.
    print(f"Rows: {len(df):,}")
    print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"Exact duplicate text rows: {int(df.duplicated('text_clean').sum()):,}")
    display(df.sample(min(rows, len(df)), random_state=SEED)[[
        "record_id", "date", "channel", "product_line", "sentiment_label",
        "complaint_label", "rating", "text"
    ]])


def display_distribution(df, column, normalize=True):
    counts = df[column].value_counts(normalize=normalize).rename("share" if normalize else "count").reset_index()
    counts.columns = [column, "share" if normalize else "count"]
    if normalize:
        counts["share"] = counts["share"].round(3)
    display(counts)


def plot_confusion(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(7, 5))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(ax=ax, values_format="d")
    ax.set_title(title)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


def evaluate_multiclass(model_name, y_true, y_pred, labels):
    row = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    print(f"{model_name} classification report")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    return row


def threshold_report(y_true, scores, thresholds):
    rows = []
    y_true = np.asarray(y_true)
    for threshold in thresholds:
        pred = (scores >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "flagged_rate": pred.mean(),
            "false_alarms": int(fp),
            "missed_cases": int(fn),
            "true_escalations": int(tp),
        })
    return pd.DataFrame(rows)


def cost_table(y_true, scores, thresholds, false_positive_cost=1.0, false_negative_cost=8.0):
    rows = []
    y_true = np.asarray(y_true)
    for threshold in thresholds:
        pred = (scores >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        cost = false_positive_cost * fp + false_negative_cost * fn
        rows.append({
            "threshold": threshold,
            "false_positive_cost": false_positive_cost,
            "false_negative_cost": false_negative_cost,
            "false_alarms": int(fp),
            "missed_cases": int(fn),
            "total_cost": float(cost),
        })
    return pd.DataFrame(rows).sort_values("total_cost")


def group_binary_metrics(df, y_col, score_col, group_col, threshold=0.50, min_n=10):
    rows = []
    for group_value, g in df.groupby(group_col):
        if len(g) < min_n:
            continue
        y_true = g[y_col].astype(int).values
        pred = (g[score_col].values >= threshold).astype(int)
        rows.append({
            group_col: group_value,
            "n": len(g),
            "base_rate": y_true.mean(),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "flagged_rate": pred.mean(),
        })
    return pd.DataFrame(rows).sort_values("f1")


def top_terms_from_logistic_pipeline(pipe, top_n=10):
    vectorizer = pipe.named_steps["tfidf"]
    clf = pipe.named_steps["clf"]
    feature_names = np.array(vectorizer.get_feature_names_out())
    coef = np.asarray(clf.coef_)
    rows = []

    # Binary logistic regression stores one coefficient vector. Positive weights
    # push toward clf.classes_[1], while negative weights push toward clf.classes_[0].
    if coef.shape[0] == 1 and len(clf.classes_) == 2:
        weights = coef[0]
        top_pos = np.argsort(weights)[::-1][:top_n]
        top_neg = np.argsort(weights)[:top_n]
        rows.append({
            "class": clf.classes_[1],
            "top_positive_terms": ", ".join(feature_names[top_pos]),
        })
        rows.append({
            "class": clf.classes_[0],
            "top_positive_terms": ", ".join(feature_names[top_neg]),
        })
        return pd.DataFrame(rows)

    for class_idx, class_name in enumerate(clf.classes_):
        weights = coef[class_idx]
        top_idx = np.argsort(weights)[::-1][:top_n]
        rows.append({
            "class": class_name,
            "top_positive_terms": ", ".join(feature_names[top_idx]),
        })
    return pd.DataFrame(rows)


def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", str(text).strip())
    return [p.strip() for p in parts if p.strip()]


def bootstrap_mean_ci(values, n_boot=500, alpha=0.05, seed=SEED):
    values = np.asarray(values, dtype=float)
    if len(values) <= 1:
        mean = float(np.mean(values)) if len(values) else np.nan
        return mean, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = [rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)]
    return float(values.mean()), float(np.quantile(boot, alpha / 2)), float(np.quantile(boot, 1 - alpha / 2))

## 2. Synthetic customer feedback corpus with metadata

The corpus below mimics a combined repository of product reviews, support tickets, chat messages, and survey comments. Each row is a text observation with metadata that business teams often use for reporting: channel, product line, customer segment, date, star rating, and operational indicators. The hidden labels are included only for classroom evaluation.

In [ ]:
# ============================================================
# 2.1 Generate synthetic customer feedback data
# ============================================================

rng = np.random.default_rng(SEED)

product_lines = ["Smart Blender", "Fitness App", "Travel Backpack", "Coffee Machine"]
channels = ["review", "support_ticket", "chat", "survey", "community"]
segments = ["new_customer", "repeat_customer", "premium", "price_sensitive"]
regions = ["West", "South", "Midwest", "Northeast"]

aspect_phrases = {
    "delivery": {
        "positive": ["delivery arrived earlier than expected", "shipping was smooth", "tracking updates were clear"],
        "negative": ["delivery was late", "the package arrived damaged", "tracking stopped updating"],
        "neutral": ["the order arrived on Tuesday", "I received a tracking number", "delivery took four days"],
    },
    "product_quality": {
        "positive": ["quality feels durable", "the product works reliably", "materials feel premium"],
        "negative": ["the product broke after one week", "quality feels cheap", "the main part stopped working"],
        "neutral": ["the product matches the description", "the box includes the listed parts", "I have used it twice"],
    },
    "app_usability": {
        "positive": ["app setup was easy", "the interface is clean", "the instructions were simple"],
        "negative": ["the app crashed during setup", "the menu is confusing", "the login screen kept freezing"],
        "neutral": ["the app asks for an email address", "the menu has five tabs", "I created an account"],
    },
    "price_billing": {
        "positive": ["the price feels fair", "the discount was clear", "the subscription options were easy to compare"],
        "negative": ["billing was confusing", "an unexpected fee appeared", "the renewal price was not clear"],
        "neutral": ["the monthly price is listed on the site", "the receipt shows tax separately", "there are two plan options"],
    },
    "customer_service": {
        "positive": ["support solved my question", "the agent was patient", "service responded quickly"],
        "negative": ["support never responded", "the agent kept transferring me", "service gave me a generic answer"],
        "neutral": ["I contacted support yesterday", "a case number was created", "the chat lasted ten minutes"],
    },
    "design": {
        "positive": ["the design looks premium", "the layout is easy to understand", "the style looks modern"],
        "negative": ["the design looks bulky", "the buttons are hard to find", "the layout feels cluttered"],
        "neutral": ["the color is black", "the design has two buttons", "the screen is on the front"],
    },
}

openers = [
    "After a week of use,",
    "For my recent order,",
    "Compared with what I expected,",
    "In my first experience,",
    "After contacting the company,",
    "For this purchase,",
]

positive_closers = [
    "I would recommend it.",
    "This made the experience easier.",
    "I am happy with the purchase.",
    "It feels like good value.",
]

negative_closers = [
    "I need this fixed soon.",
    "This made the experience frustrating.",
    "I would hesitate to buy again.",
    "Please help me resolve this.",
]

neutral_closers = [
    "I do not have enough use yet to judge it.",
    "This is only an update on what happened.",
    "I am waiting to see how it performs.",
    "No strong opinion yet.",
]

context_sentences = [
    "The order was for my household.",
    "I bought it during a promotion.",
    "This was my first purchase from the brand.",
    "I compared it with a competing option.",
    "I use it several times a week.",
    "The comment comes from a mobile app submission.",
]

sentiment_distribution = {
    "review": [0.48, 0.18, 0.19, 0.15],
    "support_ticket": [0.12, 0.55, 0.08, 0.25],
    "chat": [0.18, 0.42, 0.15, 0.25],
    "survey": [0.34, 0.25, 0.25, 0.16],
    "community": [0.35, 0.28, 0.17, 0.20],
}
sentiment_labels = ["positive", "negative", "neutral", "mixed"]

records = []
start_date = pd.Timestamp("2025-01-01")
n_records = 950

for i in range(n_records):
    channel = rng.choice(channels, p=[0.30, 0.22, 0.16, 0.20, 0.12])
    product_line = rng.choice(product_lines)
    segment = rng.choice(segments, p=[0.28, 0.34, 0.18, 0.20])
    region = rng.choice(regions)
    sentiment = rng.choice(sentiment_labels, p=sentiment_distribution[channel])
    aspect = rng.choice(list(aspect_phrases.keys()))
    opener = rng.choice(openers)
    context = rng.choice(context_sentences)

    if sentiment == "positive":
        phrase = rng.choice(aspect_phrases[aspect]["positive"])
        text = f"{opener} the {product_line} {phrase}. {context} {rng.choice(positive_closers)}"
        complaint = 0
    elif sentiment == "negative":
        phrase = rng.choice(aspect_phrases[aspect]["negative"])
        text = f"{opener} the {product_line} {phrase}. {context} {rng.choice(negative_closers)}"
        complaint = int(rng.random() < 0.88)
    elif sentiment == "neutral":
        phrase = rng.choice(aspect_phrases[aspect]["neutral"])
        text = f"{opener} the {product_line} {phrase}. {context} {rng.choice(neutral_closers)}"
        complaint = int(rng.random() < 0.04)
    else:
        pos_aspect = rng.choice(list(aspect_phrases.keys()))
        neg_aspect = rng.choice([a for a in aspect_phrases if a != pos_aspect])
        pos_phrase = rng.choice(aspect_phrases[pos_aspect]["positive"])
        neg_phrase = rng.choice(aspect_phrases[neg_aspect]["negative"])
        text = (
            f"{opener} I like that the {product_line} {pos_phrase}, "
            f"but the {neg_phrase}. {context} This leaves me unsure."
        )
        aspect = neg_aspect
        complaint = int(rng.random() < 0.58)

    if rng.random() < 0.12:
        text = text + "!"
    if rng.random() < 0.05:
        text = text.replace("frustrating", "very frustrating")
    if rng.random() < 0.03:
        text = text + " I am not happy with this part."
        if sentiment == "positive":
            sentiment = "mixed"
            complaint = int(rng.random() < 0.45)

    # Ratings are intentionally noisy proxy labels.
    rating_center = {"positive": 4.55, "negative": 1.75, "neutral": 3.05, "mixed": 3.10}[sentiment]
    rating = int(np.clip(np.rint(rng.normal(rating_center, 0.70)), 1, 5))
    if rng.random() < 0.08:
        rating = int(rng.integers(1, 6))

    prior_orders = int(np.clip(rng.poisson(3 if segment != "new_customer" else 1), 0, 15))
    delivery_days = int(np.clip(rng.normal(4.0 + (aspect == "delivery") * (1.8 if sentiment in ["negative", "mixed"] else 0), 1.5), 1, 12))
    refund_requested = int(complaint and rng.random() < (0.40 if sentiment == "negative" else 0.18))
    urgency_label = "high" if (complaint and rng.random() < (0.45 if channel in ["support_ticket", "chat"] else 0.20)) else "normal"

    segment_effect = {"new_customer": 0.30, "repeat_customer": -0.10, "premium": -0.18, "price_sensitive": 0.20}[segment]
    logit_churn = (
        -2.20
        + 1.00 * (sentiment == "negative")
        + 0.70 * (sentiment == "mixed")
        + 0.85 * complaint
        + 0.55 * refund_requested
        + 0.08 * delivery_days
        - 0.06 * prior_orders
        + segment_effect
    )
    churn_prob = 1 / (1 + math.exp(-logit_churn))
    churn_30d = int(rng.random() < churn_prob)

    date = start_date + pd.Timedelta(days=int(rng.integers(0, 182)))
    records.append({
        "record_id": f"FB-{i + 1:04d}",
        "date": date,
        "channel": channel,
        "product_line": product_line,
        "segment": segment,
        "region": region,
        "target_aspect": aspect,
        "sentiment_label": sentiment,
        "complaint_label": complaint,
        "urgency_label": urgency_label,
        "rating": rating,
        "prior_orders": prior_orders,
        "delivery_days": delivery_days,
        "refund_requested": refund_requested,
        "churn_30d": churn_30d,
        "text": text,
    })

df = pd.DataFrame(records).sort_values("date").reset_index(drop=True)
df["text_clean"] = df["text"].map(normalize_text)

# Add a small number of exact duplicate feedback records to make the quality check realistic.
dupes = df.sample(14, random_state=SEED).copy()
dupes["record_id"] = [f"FB-DUP-{i + 1:02d}" for i in range(len(dupes))]
dupes["date"] = dupes["date"] + pd.to_timedelta(rng.integers(1, 10, size=len(dupes)), unit="D")
df = pd.concat([df, dupes], ignore_index=True).sort_values("date").reset_index(drop=True)

# A leakage demonstration column. It simulates post-model routing notes that should not be used as input.
df["leaky_text"] = df["text_clean"] + np.where(
    df["complaint_label"].eq(1),
    " case status escalated by service recovery team",
    " case status closed without service recovery",
)

compact_profile(df)

In [ ]:
# ============================================================
# 2.2 Corpus profile: labels, channels, and business metadata
# ============================================================

print("Sentiment label distribution")
display_distribution(df, "sentiment_label")

print("Complaint label distribution")
display_distribution(df, "complaint_label")

profile = (
    df.groupby(["channel", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["positive", "negative", "neutral", "mixed"])
)
display(profile)

channel_volume = df["channel"].value_counts().sort_values()
plt.figure(figsize=(7, 4))
plt.bar(channel_volume.index, channel_volume.values)
plt.title("Feedback volume by channel")
plt.xlabel("Channel")
plt.ylabel("Number of records")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 3. Sentiment as a measurement task

Before choosing a model, the analyst must define what the model is supposed to measure. A broad question such as "How do customers feel about us?" has to become a measurable label. The label policy below separates general sentiment from complaint detection and urgency classification. These tasks are related, but they are not the same.

In [ ]:
# ============================================================
# 3.1 Translate business questions into measurable labels
# ============================================================

label_policy = pd.DataFrame([
    {
        "business_question": "Are customers expressing a favorable or unfavorable evaluation?",
        "analytic_task": "Sentiment classification",
        "label_set": "positive, negative, neutral, mixed",
        "unit_of_analysis": "one feedback message",
        "action_supported": "monitoring and comparison",
    },
    {
        "business_question": "Which messages indicate a problem that may require service recovery?",
        "analytic_task": "Complaint detection",
        "label_set": "complaint, not complaint",
        "unit_of_analysis": "one feedback message or support turn",
        "action_supported": "triage and escalation",
    },
    {
        "business_question": "Which cases need faster handling?",
        "analytic_task": "Urgency classification",
        "label_set": "normal, high",
        "unit_of_analysis": "one support or chat message",
        "action_supported": "routing and prioritization",
    },
])

display(label_policy)

## 4. Ratings as imperfect proxy labels

Star ratings are convenient because they are already structured, but they are not the same as sentiment in text. Ratings collapse multiple targets, expectations, and price comparisons into one number. They also cannot represent mixed text very well. The quick check below shows how much information can be lost when ratings are converted into proxy labels.

In [ ]:
# ============================================================
# 4.1 Convert ratings into proxy sentiment labels and compare with text labels
# ============================================================

rating_proxy = df.copy()
rating_proxy["rating_proxy_label"] = pd.cut(
    rating_proxy["rating"],
    bins=[0, 2, 3, 5],
    labels=["negative", "neutral", "positive"],
    include_lowest=True,
).astype(str)

proxy_table = pd.crosstab(
    rating_proxy["sentiment_label"],
    rating_proxy["rating_proxy_label"],
    normalize="index"
).round(3)

display(proxy_table)

mismatch = rating_proxy[
    rating_proxy["sentiment_label"] != rating_proxy["rating_proxy_label"]
][["record_id", "rating", "rating_proxy_label", "sentiment_label", "target_aspect", "text"]]

print(f"Proxy label mismatch rate: {len(mismatch) / len(rating_proxy):.1%}")
display(mismatch.sample(8, random_state=SEED))

## 5. Unit of analysis and target of opinion

A sentiment label is attached to a unit of analysis. That unit might be an entire review, a sentence, a chat turn, or an aspect within a sentence. The target of opinion is the object being evaluated. The same message can be positive about product design and negative about delivery. This is why a single document-level label is useful for monitoring but often too coarse for diagnosis.

In [ ]:
# ============================================================
# 5.1 The same feedback viewed at different units of analysis
# ============================================================

example_text = (
    "I love the design of the Travel Backpack. "
    "The materials feel premium, but delivery was late and support never responded. "
    "I might keep the bag, but the service experience was disappointing."
)

sentences = split_sentences(example_text)

unit_demo = pd.DataFrame([
    {
        "view": "document-level",
        "unit": example_text,
        "target": "overall experience",
        "label": "mixed",
        "managerial_use": "high-level monitoring",
    },
    {
        "view": "sentence-level",
        "unit": sentences[0],
        "target": "design",
        "label": "positive",
        "managerial_use": "retrieve positive examples",
    },
    {
        "view": "sentence-level",
        "unit": sentences[1],
        "target": "materials, delivery, support",
        "label": "mixed",
        "managerial_use": "diagnostic review",
    },
    {
        "view": "aspect-level",
        "unit": "delivery was late",
        "target": "delivery",
        "label": "negative",
        "managerial_use": "operations improvement",
    },
    {
        "view": "aspect-level",
        "unit": "support never responded",
        "target": "customer service",
        "label": "negative",
        "managerial_use": "service recovery",
    },
])

display(unit_demo)

## 6. Label agreement and data quality controls

A supervised classifier can only learn the labeling policy it receives. If annotators disagree because the label guide is unclear, model evaluation will be unstable. Agreement is not just a statistical detail. It is evidence that the organization has defined a measurable construct.

In [ ]:
# ============================================================
# 6.1 Simulate two annotators and measure agreement
# ============================================================

labels = ["positive", "negative", "neutral", "mixed"]
confusion_choices = {
    "positive": ["mixed", "neutral"],
    "negative": ["mixed", "neutral"],
    "neutral": ["positive", "negative", "mixed"],
    "mixed": ["positive", "negative", "neutral"],
}
correct_prob = {"positive": 0.93, "negative": 0.91, "neutral": 0.76, "mixed": 0.69}


def simulate_annotator(true_label, rng):
    if rng.random() < correct_prob[true_label]:
        return true_label
    return rng.choice(confusion_choices[true_label])

annotation_sample = df.sample(180, random_state=SEED).copy()
ann_rng = np.random.default_rng(SEED + 10)
annotation_sample["annotator_A"] = [simulate_annotator(x, ann_rng) for x in annotation_sample["sentiment_label"]]
annotation_sample["annotator_B"] = [simulate_annotator(x, ann_rng) for x in annotation_sample["sentiment_label"]]

kappa = cohen_kappa_score(annotation_sample["annotator_A"], annotation_sample["annotator_B"], labels=labels)
print(f"Cohen's kappa between two simulated annotators: {kappa:.3f}")

agreement_table = pd.crosstab(annotation_sample["annotator_A"], annotation_sample["annotator_B"])
display(agreement_table)

disagreements = annotation_sample[annotation_sample["annotator_A"] != annotation_sample["annotator_B"]][[
    "record_id", "sentiment_label", "annotator_A", "annotator_B", "text"
]]
print(f"Disagreement rate: {len(disagreements) / len(annotation_sample):.1%}")
display(disagreements.sample(8, random_state=SEED))

In [ ]:
# ============================================================
# 6.2 Deduplicate and create a time-respecting train-test split
# ============================================================

# In a real project, this step should happen before model evaluation.
df_model = df.drop_duplicates("text_clean").sort_values("date").reset_index(drop=True)

cutoff = pd.Timestamp("2025-05-15")
train_df = df_model[df_model["date"] < cutoff].copy()
test_df = df_model[df_model["date"] >= cutoff].copy()

print(f"Modeling rows after exact-text deduplication: {len(df_model):,}")
print(f"Train rows before {cutoff.date()}: {len(train_df):,}")
print(f"Test rows on or after {cutoff.date()}: {len(test_df):,}")

split_profile = pd.concat({
    "train": train_df["sentiment_label"].value_counts(normalize=True),
    "test": test_df["sentiment_label"].value_counts(normalize=True),
}, axis=1).fillna(0).round(3)
display(split_profile.reindex(labels))

quality_checklist = pd.DataFrame([
    {"control": "Clear label definitions", "status": "classroom policy defined above", "risk_reduced": "construct drift"},
    {"control": "Agreement review", "status": f"kappa = {kappa:.3f} on pilot labels", "risk_reduced": "hidden ambiguity"},
    {"control": "Duplicate check", "status": f"removed {len(df) - len(df_model)} exact duplicate rows", "risk_reduced": "test leakage"},
    {"control": "Time-respecting split", "status": f"cutoff = {cutoff.date()}", "risk_reduced": "future information leakage"},
    {"control": "Slice coverage", "status": "channel and product metadata retained", "risk_reduced": "uneven performance"},
])
display(quality_checklist)

## 7. Lexicon and rule-based sentiment baseline

A lexicon baseline is transparent and easy to audit. It will not handle every context, but it gives the team a useful reference point before moving to supervised models. The function below uses small positive and negative word lists, checks for negation, and assigns a mixed label when both positive and negative evidence appear.

In [ ]:
# ============================================================
# 7.1 A simple lexicon baseline with negation and mixed-evidence handling
# ============================================================

positive_terms = {
    "happy", "recommend", "easy", "smooth", "clear", "durable", "reliably", "premium",
    "fair", "solved", "patient", "quickly", "modern", "good", "easier", "value",
}
negative_terms = {
    "late", "damaged", "stopped", "broke", "cheap", "crashed", "confusing", "freezing",
    "unexpected", "generic", "bulky", "cluttered", "frustrating", "hesitate", "disappointing",
    "not", "never", "hard",
}
negators = {"not", "never", "no", "hardly", "barely"}
contrast_terms = {"but", "however", "although", "though"}


def lexicon_sentiment(text):
    tokens = simple_tokenize(text)
    pos_hits = 0
    neg_hits = 0
    for idx, tok in enumerate(tokens):
        previous = tokens[max(0, idx - 2):idx]
        is_negated = any(prev in negators for prev in previous)
        if tok in positive_terms:
            if is_negated:
                neg_hits += 1
            else:
                pos_hits += 1
        if tok in negative_terms:
            if is_negated and tok != "not":
                pos_hits += 1
            else:
                neg_hits += 1

    has_contrast = any(tok in contrast_terms for tok in tokens)
    if (pos_hits > 0 and neg_hits > 0) or (has_contrast and (pos_hits + neg_hits) > 0):
        return "mixed"
    if pos_hits > neg_hits:
        return "positive"
    if neg_hits > pos_hits:
        return "negative"
    return "neutral"

lexicon_pred = test_df["text_clean"].map(lexicon_sentiment)
lexicon_result = evaluate_multiclass("lexicon_baseline", test_df["sentiment_label"], lexicon_pred, labels)
plot_confusion(test_df["sentiment_label"], lexicon_pred, labels, "Lexicon baseline confusion matrix")

In [ ]:
# ============================================================
# 7.2 Inspect lexicon errors
# ============================================================

lexicon_errors = test_df.copy()
lexicon_errors["lexicon_pred"] = lexicon_pred.values
lexicon_errors = lexicon_errors[lexicon_errors["sentiment_label"] != lexicon_errors["lexicon_pred"]][[
    "record_id", "channel", "sentiment_label", "lexicon_pred", "rating", "text"
]]

print(f"Lexicon errors in test set: {len(lexicon_errors):,}")
display(lexicon_errors.sample(min(10, len(lexicon_errors)), random_state=SEED))

## 8. Supervised text classification with sparse features

Sparse features such as TF-IDF remain strong baselines for business text classification. They are fast, interpretable, and easy to deploy. The key rule is to fit the vectorizer only on the training set, then transform the test set. Fitting the vectorizer on all data would leak information from the evaluation period.

In [ ]:
# ============================================================
# 8.1 TF-IDF plus logistic regression for four-class sentiment classification
# ============================================================

sentiment_sparse = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.90,
        stop_words="english",
    )),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
])

sentiment_sparse.fit(train_df["text_clean"], train_df["sentiment_label"])
sparse_pred = sentiment_sparse.predict(test_df["text_clean"])
sparse_result = evaluate_multiclass("tfidf_logistic", test_df["sentiment_label"], sparse_pred, labels)
plot_confusion(test_df["sentiment_label"], sparse_pred, labels, "TF-IDF logistic sentiment classifier")

In [ ]:
# ============================================================
# 8.2 Inspect terms that push the sparse classifier toward each class
# ============================================================

top_terms = top_terms_from_logistic_pipeline(sentiment_sparse, top_n=12)
display(top_terms)

model_comparison = pd.DataFrame([lexicon_result, sparse_result]).round(3)
display(model_comparison)

## 9. Dense representation baseline

Dense representations compress text into a smaller vector space. In later chapters, dense vectors may come from pre-trained language models. To keep this notebook lightweight and offline, we approximate the idea with a TF-IDF matrix followed by truncated singular value decomposition. This is not a replacement for BERT or other transformer models, but it helps illustrate the shift from sparse word features to dense document vectors.

In [ ]:
# ============================================================
# 9.1 TF-IDF plus SVD document vectors plus logistic regression
# ============================================================

dense_sentiment = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.90,
        stop_words="english",
    )),
    ("svd", TruncatedSVD(n_components=30, random_state=SEED)),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
])

dense_sentiment.fit(train_df["text_clean"], train_df["sentiment_label"])
dense_pred = dense_sentiment.predict(test_df["text_clean"])
dense_result = evaluate_multiclass("svd_dense_logistic", test_df["sentiment_label"], dense_pred, labels)
plot_confusion(test_df["sentiment_label"], dense_pred, labels, "Dense SVD sentiment classifier")

model_comparison = pd.DataFrame([lexicon_result, sparse_result, dense_result]).round(3)
display(model_comparison)

In [ ]:
# ============================================================
# 9.2 Visualize the first two dense dimensions as a diagnostic, not as proof
# ============================================================

viz_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_df=0.90, stop_words="english")),
    ("svd", TruncatedSVD(n_components=2, random_state=SEED)),
])

points = viz_pipe.fit_transform(df_model["text_clean"])
label_codes = pd.Categorical(df_model["sentiment_label"], categories=labels).codes

plt.figure(figsize=(7, 5))
plt.scatter(points[:, 0], points[:, 1], c=label_codes, alpha=0.55)
plt.title("Two-dimensional SVD view of feedback texts")
plt.xlabel("Dense dimension 1")
plt.ylabel("Dense dimension 2")
plt.tight_layout()
plt.show()

print("Caution: a two-dimensional projection is useful for diagnostics, but it is not a full validation of model quality.")

## 10. Evaluation that matches business risk

For operational use, a sentiment label is often less important than a risk flag. A service team may want to detect complaint messages that should be reviewed or escalated. In that setting, the threshold is a policy choice. Lower thresholds catch more true complaints but create more workload. Higher thresholds reduce workload but miss more cases.

In [ ]:
# ============================================================
# 10.1 Binary complaint detection model and threshold table
# ============================================================

complaint_sparse = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.90,
        stop_words="english",
    )),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
])

complaint_sparse.fit(train_df["text_clean"], train_df["complaint_label"])
complaint_scores = complaint_sparse.predict_proba(test_df["text_clean"])[:, 1]
complaint_pred_50 = (complaint_scores >= 0.50).astype(int)

print("Complaint detection at threshold = 0.50")
print(classification_report(test_df["complaint_label"], complaint_pred_50, target_names=["not_complaint", "complaint"], zero_division=0))
plot_confusion(test_df["complaint_label"], complaint_pred_50, [0, 1], "Complaint detection confusion matrix at 0.50")

thresholds = np.round(np.arange(0.10, 0.91, 0.10), 2)
thresh_df = threshold_report(test_df["complaint_label"], complaint_scores, thresholds).round(3)
display(thresh_df)

In [ ]:
# ============================================================
# 10.2 Precision-recall curve and cost-sensitive threshold selection
# ============================================================

precision, recall, pr_thresholds = precision_recall_curve(test_df["complaint_label"], complaint_scores)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, marker="o", markersize=3)
plt.title("Precision-recall trade-off for complaint detection")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.tight_layout()
plt.show()

business_costs = cost_table(
    test_df["complaint_label"],
    complaint_scores,
    thresholds=np.round(np.arange(0.10, 0.91, 0.05), 2),
    false_positive_cost=1.0,
    false_negative_cost=8.0,
)

display(business_costs.head(8).round(2))
best_threshold = float(business_costs.iloc[0]["threshold"])
print(f"Cost-minimizing threshold under this policy: {best_threshold:.2f}")

In [ ]:
# ============================================================
# 10.3 Calibration check: do probability scores behave like probabilities?
# ============================================================

prob_true, prob_pred = calibration_curve(test_df["complaint_label"], complaint_scores, n_bins=6, strategy="uniform")
brier = brier_score_loss(test_df["complaint_label"], complaint_scores)
print(f"Brier score: {brier:.3f}")

plt.figure(figsize=(6, 5))
plt.plot(prob_pred, prob_true, marker="o")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Calibration curve for complaint probability")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed complaint rate")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 10.4 Convert probabilities into a three-zone triage policy
# ============================================================

low_threshold = 0.35
high_threshold = 0.75

def triage_action(score):
    if score >= high_threshold:
        return "auto_escalate"
    if score >= low_threshold:
        return "human_review"
    return "monitor_only"

triage_df = test_df.copy()
triage_df["complaint_score"] = complaint_scores
triage_df["triage_action"] = triage_df["complaint_score"].map(triage_action)

action_summary = (
    triage_df.groupby("triage_action")
    .agg(
        n=("record_id", "count"),
        complaint_rate=("complaint_label", "mean"),
        mean_score=("complaint_score", "mean"),
    )
    .sort_values("mean_score", ascending=False)
    .round(3)
)
display(action_summary)

display(triage_df.sort_values("complaint_score", ascending=False)[[
    "record_id", "channel", "product_line", "complaint_label", "complaint_score", "triage_action", "text"
]].head(10))

## 11. Slice evaluation, error review, and leakage audit

Overall performance can hide failures in specific channels, product lines, regions, or customer groups. A reliable text classification system should be evaluated by relevant slices and reviewed through concrete errors. It should also be audited for leakage, especially when text has been combined with service notes, routing labels, or post-outcome fields.

In [ ]:
# ============================================================
# 11.1 Slice-level complaint detection metrics
# ============================================================

test_eval = test_df.copy()
test_eval["complaint_score"] = complaint_scores

print("By channel")
display(group_binary_metrics(test_eval, "complaint_label", "complaint_score", "channel", threshold=best_threshold, min_n=10).round(3))

print("By product line")
display(group_binary_metrics(test_eval, "complaint_label", "complaint_score", "product_line", threshold=best_threshold, min_n=10).round(3))

print("By segment")
display(group_binary_metrics(test_eval, "complaint_label", "complaint_score", "segment", threshold=best_threshold, min_n=10).round(3))

In [ ]:
# ============================================================
# 11.2 Error review at the chosen threshold
# ============================================================

test_eval["complaint_pred"] = (test_eval["complaint_score"] >= best_threshold).astype(int)
test_eval["error_type"] = np.select(
    [
        (test_eval["complaint_label"].eq(1) & test_eval["complaint_pred"].eq(1)),
        (test_eval["complaint_label"].eq(0) & test_eval["complaint_pred"].eq(1)),
        (test_eval["complaint_label"].eq(1) & test_eval["complaint_pred"].eq(0)),
        (test_eval["complaint_label"].eq(0) & test_eval["complaint_pred"].eq(0)),
    ],
    ["true_positive", "false_positive", "false_negative", "true_negative"],
    default="unknown",
)

error_counts = test_eval["error_type"].value_counts().rename_axis("error_type").reset_index(name="count")
display(error_counts)

for error_type in ["false_positive", "false_negative"]:
    print(f"\nExamples: {error_type}")
    examples = test_eval[test_eval["error_type"] == error_type].sort_values("complaint_score", ascending=False)
    if len(examples) == 0:
        print("No examples in this synthetic run.")
    else:
        display(examples[["record_id", "channel", "sentiment_label", "complaint_label", "complaint_score", "target_aspect", "text"]].head(6))

In [ ]:
# ============================================================
# 11.3 Leakage audit: post-outcome routing notes inflate evaluation
# ============================================================

leaky_model = clone(complaint_sparse)
leaky_model.fit(train_df["leaky_text"], train_df["complaint_label"])
leaky_scores = leaky_model.predict_proba(test_df["leaky_text"])[:, 1]
leaky_pred = (leaky_scores >= 0.50).astype(int)

clean_pred = (complaint_scores >= 0.50).astype(int)
leakage_comparison = pd.DataFrame([
    {
        "input_text": "clean customer text only",
        "precision": precision_score(test_df["complaint_label"], clean_pred, zero_division=0),
        "recall": recall_score(test_df["complaint_label"], clean_pred, zero_division=0),
        "f1": f1_score(test_df["complaint_label"], clean_pred, zero_division=0),
        "auc": roc_auc_score(test_df["complaint_label"], complaint_scores),
    },
    {
        "input_text": "customer text plus post-outcome routing note",
        "precision": precision_score(test_df["complaint_label"], leaky_pred, zero_division=0),
        "recall": recall_score(test_df["complaint_label"], leaky_pred, zero_division=0),
        "f1": f1_score(test_df["complaint_label"], leaky_pred, zero_division=0),
        "auc": roc_auc_score(test_df["complaint_label"], leaky_scores),
    },
]).round(3)

display(leakage_comparison)

leakage_terms = top_terms_from_logistic_pipeline(leaky_model, top_n=12)
display(leakage_terms)

print("The suspicious terms in the leaky model are not customer language. They are process labels added after the outcome.")

## 12. Using sentiment for dashboards and downstream analytics

A classifier produces predictions for individual texts. Managers usually act on products, channels, regions, cohorts, and time periods. Aggregation is therefore part of the measurement design. A good dashboard reports the level of sentiment, message volume, uncertainty, and representative examples.

In [ ]:
# ============================================================
# 12.1 Aggregate message-level sentiment into a weekly business indicator
# ============================================================

all_probs = sentiment_sparse.predict_proba(df_model["text_clean"])
class_order = list(sentiment_sparse.named_steps["clf"].classes_)
positive_idx = class_order.index("positive")
negative_idx = class_order.index("negative")

dashboard_df = df_model.copy()
dashboard_df["sentiment_index"] = all_probs[:, positive_idx] - all_probs[:, negative_idx]
dashboard_df["week"] = dashboard_df["date"].dt.to_period("W").dt.start_time

weekly_rows = []
for week, g in dashboard_df.groupby("week"):
    mean, lo, hi = bootstrap_mean_ci(g["sentiment_index"].values, n_boot=300, seed=SEED)
    weekly_rows.append({"week": week, "n": len(g), "sentiment_index": mean, "lo": lo, "hi": hi})
weekly = pd.DataFrame(weekly_rows).sort_values("week")

display(weekly.head())

plt.figure(figsize=(9, 4))
plt.plot(weekly["week"], weekly["sentiment_index"], marker="o")
plt.fill_between(weekly["week"], weekly["lo"], weekly["hi"], alpha=0.20)
plt.title("Weekly sentiment index with uncertainty band")
plt.xlabel("Week")
plt.ylabel("P(positive) - P(negative)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 3))
plt.bar(weekly["week"], weekly["n"], width=5)
plt.title("Weekly message volume")
plt.xlabel("Week")
plt.ylabel("Number of messages")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 12.2 Downstream churn prediction: structured-only versus structured plus text signal
# ============================================================

# Use cross-fitted complaint scores for the training set to reduce overfitting in the downstream model.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
train_text_signal = cross_val_predict(
    complaint_sparse,
    train_df["text_clean"],
    train_df["complaint_label"],
    cv=cv,
    method="predict_proba",
)[:, 1]

complaint_sparse.fit(train_df["text_clean"], train_df["complaint_label"])
test_text_signal = complaint_sparse.predict_proba(test_df["text_clean"])[:, 1]

base_features = ["rating", "prior_orders", "delivery_days", "refund_requested"]
X_train_base = train_df[base_features].astype(float).values
X_test_base = test_df[base_features].astype(float).values

y_train_churn = train_df["churn_30d"].astype(int).values
y_test_churn = test_df["churn_30d"].astype(int).values

structured_model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
])
structured_model.fit(X_train_base, y_train_churn)
structured_auc = roc_auc_score(y_test_churn, structured_model.predict_proba(X_test_base)[:, 1])

X_train_text = np.column_stack([X_train_base, train_text_signal])
X_test_text = np.column_stack([X_test_base, test_text_signal])

text_enhanced_model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
])
text_enhanced_model.fit(X_train_text, y_train_churn)
text_enhanced_auc = roc_auc_score(y_test_churn, text_enhanced_model.predict_proba(X_test_text)[:, 1])

auc_comparison = pd.DataFrame([
    {"downstream_model": "structured_only", "features": ", ".join(base_features), "test_auc": structured_auc},
    {"downstream_model": "structured_plus_text", "features": ", ".join(base_features + ["complaint_score"]), "test_auc": text_enhanced_auc},
]).round(3)

display(auc_comparison)

print("Interpretation: sentiment-derived features are useful only if they add incremental value and pass slice-level bias and drift checks.")

## 13. Governance output: sentiment classification system card

A sentiment system card documents the intended use, label policy, inputs, evaluation, known limitations, and monitoring plan. It helps prevent a model from being reused for decisions it was not designed to support.

In [ ]:
# ============================================================
# 13.1 Create a compact sentiment system card
# ============================================================

system_card = pd.DataFrame([
    {"field": "Intended use", "entry": "Monitor feedback sentiment and triage likely complaint messages."},
    {"field": "Text unit", "entry": "One customer feedback message after exact-text deduplication."},
    {"field": "Primary labels", "entry": "Sentiment: positive, negative, neutral, mixed. Complaint: complaint or not complaint."},
    {"field": "Training split", "entry": f"Time-respecting split with cutoff {cutoff.date()}."},
    {"field": "Baseline", "entry": "Lexicon sentiment baseline retained for transparency and audit."},
    {"field": "Operational threshold", "entry": f"Complaint threshold selected by cost policy: {best_threshold:.2f}."},
    {"field": "Required monitoring", "entry": "Track performance by channel, product line, segment, and reporting period."},
    {"field": "Known limitations", "entry": "Synthetic demonstration data, no sarcasm benchmark, no multilingual coverage, no real privacy risk assessment."},
    {"field": "Misuse warning", "entry": "Do not treat sentiment as a direct measure of satisfaction, loyalty, or causal impact."},
])

display(system_card)

system_card.to_csv(OUTPUT_DIR / "ch23_sentiment_system_card.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'ch23_sentiment_system_card.csv'}")

## Exercises

1. Change the complaint threshold policy by setting a higher false-negative cost. How does the selected threshold change?
2. Redefine the sentiment task as three classes by merging `mixed` and `neutral`. Does performance improve, and what information is lost?
3. Train a separate classifier for `urgency_label`. Which evaluation metric is most relevant for a service team?
4. Evaluate complaint detection separately for support tickets and product reviews. Which channel is harder?
5. Create an aspect-level dashboard by grouping sentiment predictions by `target_aspect` and week.
6. Add a small set of sarcastic examples such as "Great, another broken login screen" and inspect model errors.
7. Remove the rating feature from the downstream churn model. Does the text-derived signal become more valuable?
8. Write a short paragraph explaining why the leakage audit in Section 11.3 is a governance problem rather than just a technical mistake.

In [ ]:
# ============================================================
# Optional exercise starter: merge mixed and neutral into one middle category
# ============================================================

exercise_df = df_model.copy()
exercise_df["sentiment_3class"] = exercise_df["sentiment_label"].replace({"mixed": "middle", "neutral": "middle"})

exercise_train = exercise_df[exercise_df["date"] < cutoff].copy()
exercise_test = exercise_df[exercise_df["date"] >= cutoff].copy()

exercise_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_df=0.90, stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
])

exercise_model.fit(exercise_train["text_clean"], exercise_train["sentiment_3class"])
exercise_pred = exercise_model.predict(exercise_test["text_clean"])

print("Three-class exercise report")
print(classification_report(exercise_test["sentiment_3class"], exercise_pred, zero_division=0))